In [ ]:
import os

# 1. Create the hidden folder inside Colab
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

# 2. Write your new token string directly into the required file
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write('<REDACTED - token was revoked>')

# 3. Secure the file permissions
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)

print("Kaggle setup complete! You can now run your download commands.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle competitions download -c equity-post-HCT-survival-predictions



In [ ]:
!unzip equity-post-HCT-survival-predictions.zip -d equity-post-HCT-survival-predictions


In [ ]:
!pip install lifelines
!pip install autograd
!pip install formulaic


In [ ]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

In [ ]:
"""
To evaluate the equitable prediction of transplant survival outcomes,
we use the concordance index (C-index) between a series of event
times and a predicted score across each race group.

It represents the global assessment of the model discrimination power:
this is the model’s ability to correctly provide a reliable ranking
of the survival times based on the individual risk scores.

The concordance index is a value between 0 and 1 where:

0.5 is the expected result from random predictions,
1.0 is perfect concordance (with no censoring, otherwise <1.0),
0.0 is perfect anti-concordance (with no censoring, otherwise >0.0)

"""

import pandas as pd
import pandas.api.types
import numpy as np
from lifelines.utils import concordance_index

class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    >>> import pandas as pd
    >>> row_id_column_name = "id"
    >>> y_pred = {'prediction': {0: 1.0, 1: 0.0, 2: 1.0}}
    >>> y_pred = pd.DataFrame(y_pred)
    >>> y_pred.insert(0, row_id_column_name, range(len(y_pred)))
    >>> y_true = { 'efs': {0: 1.0, 1: 0.0, 2: 0.0}, 'efs_time': {0: 25.1234,1: 250.1234,2: 2500.1234}, 'race_group': {0: 'race_group_1', 1: 'race_group_1', 2: 'race_group_1'}}
    >>> y_true = pd.DataFrame(y_true)
    >>> y_true.insert(0, row_id_column_name, range(len(y_true)))
    >>> score(y_true.copy(), y_pred.copy(), row_id_column_name)
    0.75
    """

    del solution[row_id_column_name]
    del submission[row_id_column_name]

    event_label = 'efs'
    interval_label = 'efs_time'
    prediction_label = 'prediction'
    for col in submission.columns:
        if not pandas.api.types.is_numeric_dtype(submission[col]):
            raise ParticipantVisibleError(f'Submission column {col} must be a number')
    # Merging solution and submission dfs on ID
    merged_df = pd.concat([solution, submission], axis=1)
    merged_df.reset_index(inplace=True)
    merged_df_race_dict = dict(merged_df.groupby(['race_group']).groups)
    metric_list = []
    for race in merged_df_race_dict.keys():
        # Retrieving values from y_test based on index
        indices = sorted(merged_df_race_dict[race])
        merged_df_race = merged_df.iloc[indices]
        # Calculate the concordance index
        c_index_race = concordance_index(
                        merged_df_race[interval_label],
                        -merged_df_race[prediction_label],
                        merged_df_race[event_label])
        metric_list.append(c_index_race)
    return float(np.mean(metric_list)-np.sqrt(np.var(metric_list)))

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

In [ ]:
test = pd.read_csv("/test.csv")
print("Test shape:", test.shape )


train = pd.read_csv("/train.csv")
print("Train shape:",train.shape)

In [ ]:
from lifelines import KaplanMeierFitter, NelsonAalenFitter

def transform_kmf(df, time_col='efs_time', event_col='efs'):
    """
    Transform using survival probability estimates
    """
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    y = kmf.survival_function_at_times(df[time_col]).values
    return y


In [ ]:
train['y'] = transform_kmf(train, 'efs_time', 'efs')

# try adding -0.1
train.loc[train['efs'] == 0, 'y'] -= 0.1

In [ ]:
to_rmv =['ID', 'efs', 'efs_time', 'y']
features = [col for col in train.columns if not col in to_rmv]
cat = [c for c in features if train[c].dtype == 'object']
num = [c for c in features if train[c].dtype != 'object']
target = 'y'
print(f'There are {len(features)} features')
print(f'There are {len(cat)} for catagorial and {len(num)} for numerical')

*italicised text*# Feature eng

In [ ]:
for c in cat:
    train[c].fillna('Missing', inplace=True)
    test[c].fillna('Missing', inplace=True)

In [ ]:
def convert_64_to_32(df, num_features):
    for c in num_features:
        if df[c].dtype == 'float64':
            df[c] = df[c].astype('float32')
        else:
            df[c] = df[c].astype('int32')
    return df

train = convert_64_to_32(train, num)
test = convert_64_to_32(test, num)

In [ ]:
def clean_columns(df):

    value_mappings = {
        'cmv_status': {
            '+/+': 'Positive_Positive',
            '+/-': 'Positive_Negative',
            '-/+': 'Negative_Positive',
            '-/-': 'Negative_Negative'
        },
        'tbi_status': {
            'No TBI': 'No_Total_Body_Irradiation',
            'TBI + Cy +- Other': 'TBI_with_Cyclophosphamide_and_Other',
            'TBI +- Other, <=cGy': 'TBI_with_Other_Low_Dose',
            'TBI +- Other, >cGy': 'TBI_with_Other_High_Dose',
            'TBI +- Other, -cGy, single': 'TBI_with_Other_Single_Dose',
            'TBI +- Other, unknown dose': 'TBI_with_Other_Unknown_Dose',
            'TBI +- Other, -cGy, unknown dose': 'TBI_with_Other_Unknown_Dose',
            'TBI +- Other, -cGy, fractionated': 'TBI_with_Other_Fractionated_Dose'
        },
        'dri_score': {
            'Intermediate': 'Intermediate_Risk',
            'N/A - pediatric': 'Not_Applicable_Pediatric',
            'High': 'High_Risk',
            'N/A - non-malignant indication': 'Not_Applicable_Non_Malignant',
            'TBD cytogenetics': 'To_Be_Determined_Cytogenetics',
            'Low': 'Low_Risk',
            'High - TED AML case <missing cytogenetics': 'High_Risk_TED_AML_Missing_Cytogenetics',
            'Intermediate - TED AML case <missing cytogenetics': 'Intermediate_Risk_TED_AML_Missing_Cytogenetics',
            'N/A - disease not classifiable': 'Not_Applicable_Disease_Not_Classifiable',
            'Very high': 'Very_High_Risk',
            'Missing disease status': 'Missing'
        },
        'tce_imm_match': {
            'P/P': 'Perfect_Perfect',
            'G/G': 'Good_Good',
            'H/H': 'Heterozygous_Heterozygous',
            'G/B': 'Good_Bad',
            'H/B': 'Heterozygous_Bad',
            'P/H': 'Perfect_Heterozygous',
            'P/B': 'Perfect_Bad',
            'P/G': 'Perfect_Good'
        },
        'gvhd_proph': {
            'FK+ MMF +- others': 'Tacrolimus_MMF_with_Others',
            'Cyclophosphamide alone': 'Cyclophosphamide_Alone',
            'FK+ MTX +- others(not MMF)': 'Tacrolimus_MTX_with_Others_Not_MMF',
            'Cyclophosphamide +- others': 'Cyclophosphamide_with_Others',
            'CSA + MMF +- others(not FK)': 'Cyclosporine_MMF_with_Others_Not_Tacrolimus',
            'FKalone': 'Tacrolimus_Alone',
            'Other GVHD Prophylaxis': 'Other_GVHD_Prophylaxis',
            'TDEPLETION alone': 'TCell_Depletion_Alone',
            'TDEPLETION +- other': 'TCell_Depletion_with_Others',
            'No GvHD Prophylaxis': 'No_GVHD_Prophylaxis',
            'CDselect alone': 'CD_Selection_Alone',
            'CSA + MTX +- others(not MMF,FK)': 'Cyclosporine_MTX_with_Others_Not_MMF_Tacrolimus',
            'CSA alone': 'Cyclosporine_Alone',
            'Parent Q = yes, but no agent': 'Parent_Yes_No_Agent',
            'CDselect +- other': 'CD_Selection_with_Others',
            'CSA +- others(not FK,MMF,MTX)': 'Cyclosporine_with_Others_Not_Tacrolimus_MMF_MTX',
            'FK+- others(not MMF,MTX)': 'Tacrolimus_with_Others_Not_MMF_MTX'
        }
    }

    for col, mappings in value_mappings.items():
        if col in df.columns:
            df[col] = df[col].astype(str).map(mappings).fillna(df[col])

    return df

def clean_space(df):
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: x.replace(' ', '_') if isinstance(x, str) else x)
    return df

def clean_not_done(df):
    df = df.map(lambda x: 'Missing' if x == 'Not done' else x)
    return df

def clean_not_tested(df):
    df = df.map(lambda x: 'Missing' if x == 'Not tested' else x)
    return df

def clean_dri_score(score):
    if isinstance(score, str) and 'Missing disease status' in score:
        return 'Missing'
    return score

def clean_conditioning_intensity(score):
    if isinstance(score, str) and 'No drugs reported' in score:
        return 'Missing'
    return score

In [ ]:
train = clean_columns(train)
test = clean_columns(test)

# train = clean_not_done(train)
# test = clean_not_done(test)

# train = clean_not_tested(train)
# test = clean_not_tested(test)

train['dri_score'] = train['dri_score'].apply(clean_dri_score)
test['dri_score'] = test['dri_score'].apply(clean_dri_score)

train['conditioning_intensity'] = train['conditioning_intensity'].apply(clean_conditioning_intensity)
test['conditioning_intensity'] = test['conditioning_intensity'].apply(clean_conditioning_intensity)

train = clean_space(train)
test = clean_space(test)

# LightGBM

In [ ]:
from sklearn.preprocessing import LabelEncoder


def encode_features_lgb(train, test, cat, num):
    train = train.copy()
    test = test.copy()

    for c in cat:
        encoder = LabelEncoder()

        # Fit and transform the training data
        train[c] = train[c].astype(str)
        train[c] = encoder.fit_transform(train[c])
        train[c] = train[c].astype('int32').astype('category')

        # Transform the test data using the encoder fitted on the training data
        test[c] = test[c].astype(str)
        test[c] = encoder.transform(test[c])
        test[c] = test[c].astype('int32').astype('category')

    return train, test

train_lgb, test_lgb = encode_features_lgb(train, test, cat, num)

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split, KFold
from sklearn.metrics import *
import lightgbm as lgb
from lightgbm import LGBMRegressor

def train_lgbm(train, test, model_params, features, cat_features, target):

    fix_params = {
        'n_estimators': 10000,
        'objective': 'regression',
        'early_stopping_rounds':20,
        'verbose': -1
    }
    model_params.update(fix_params)

    FOLDS = 10
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_lgb = np.zeros(len(train))
    pred_lgb = np.zeros(len(test))

    print('Training model with the followin parameters')
    for k, v in model_params.items():
        print(f'{k} : {v}')

    for i, (t_idx, v_idx) in enumerate(kf.split(train)):

        X_train = train.iloc[t_idx][features].copy()
        y_train = train.iloc[t_idx][target]
        X_valid = train.iloc[v_idx][features].copy()
        y_valid = train.iloc[v_idx][target]
        X_test = test[features].copy()

        model_lgb = LGBMRegressor(**model_params)
        model_lgb.fit(
            X_train, np.log1p(y_train),
            eval_set=[(X_valid, np.log1p(y_valid))],
        )

        y_valid_preds = np.expm1(model_lgb.predict(X_valid))
        oof_lgb[v_idx] = y_valid_preds
        pred_lgb += np.expm1(model_lgb.predict(X_test))


        fold_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_preds))
        print("#"*25)
        print(f"### Fold {i+1} \n")
        print(f"Fold {i+1} RMSE: {fold_rmse}")
        print("#"*25)

    pred_lgb /= FOLDS

    return model_lgb, oof_lgb, pred_lgb


# XGBoost

In [ ]:
def encode_features_xgb(train, test, label_encode_features, one_hot_encode_features, features):

    # Label Encoding
    label_encoders = {}
    for feature in label_encode_features:
        encoder = LabelEncoder()
        train[feature] = train[feature].astype(str)
        train[feature] = encoder.fit_transform(train[feature])
        test[feature] = test[feature].astype(str)
        test[feature] = encoder.transform(test[feature])
        label_encoders[feature] = encoder

    # One-Hot Encoding
    train_one_hot = pd.get_dummies(train[one_hot_encode_features], prefix=one_hot_encode_features)
    test_one_hot = pd.get_dummies(test[one_hot_encode_features], prefix=one_hot_encode_features)

    train_one_hot, test_one_hot = train_one_hot.align(test_one_hot, join="outer", axis=1, fill_value=0)

    train_xgb = pd.concat([train, train_one_hot], axis=1)
    test_xgb = pd.concat([test, test_one_hot], axis=1)

    features_xgb = features.copy()
    features_xgb.extend(train_one_hot.columns)
    features_xgb = [f for f in features_xgb if f not in one_hot_encode_features]

    return train_xgb, test_xgb, features_xgb

In [ ]:
# Define features for label and one-hot encoding
label_encode_features = [
    "dri_score", "psych_disturb", "cyto_score", "diabetes",
    "arrhythmia", "vent_hist", "renal_issue", "pulm_severe",
    "cmv_status", "tce_imm_match", "rituximab", "cyto_score_detail",
    "conditioning_intensity", "ethnicity", "obesity", "mrd_hct",
    "in_vivo_tcd", "tce_match", "hepatic_severe", "prior_tumor",
    "peptic_ulcer", "gvhd_proph", "rheum_issue", "sex_match",
    "hepatic_mild", "tce_div_match", "donor_related", "melphalan_dose",
    "cardiac", "pulm_moderate"
]

one_hot_encode_features = [
    "tbi_status", "graft_type", "prod_type", "prim_disease_hct", "race_group"
]

# Apply the encoding
train_xgb, test_xgb, features_xgb = encode_features_xgb(train, test, label_encode_features, one_hot_encode_features, features)


In [ ]:
def object_to_cat(df):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    return df
train_xgb2 = object_to_cat(train)
test_xgb2 = object_to_cat(test)

In [ ]:
from xgboost import XGBRegressor

def train_XGB(train, test, model_params, features, target):
    fix_params = {
        'tree_method': 'hist',
        'device': 'cuda',
        'objective': 'reg:squarederror',
        'n_estimators': 10000,
        'early_stopping_rounds': 20,
        'eval_metric': 'rmse',
        'enable_categorical': True
    }
    model_params.update(fix_params)

    FOLDS = 10
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_xgb = np.zeros(len(train))
    pred_xgb = np.zeros(len(test))

    print('Training XGBoost model with the following parameters:')
    for k, v in model_params.items():
        print(f'{k} : {v}')

    for i, (t_idx, v_idx) in enumerate(kf.split(train)):
        X_train = train.iloc[t_idx][features].copy()
        y_train = train.iloc[t_idx][target]
        X_valid = train.iloc[v_idx][features].copy()
        y_valid = train.iloc[v_idx][target]
        X_test = test[features].copy()

        model_xgb = XGBRegressor(**model_params)
        model_xgb.fit(
            X_train, np.log1p(y_train),
            eval_set=[(X_valid, np.log1p(y_valid))],
            verbose=False
        )

        y_valid_preds = np.expm1(model_xgb.predict(X_valid))
        oof_xgb[v_idx] = y_valid_preds
        pred_xgb += np.expm1(model_xgb.predict(X_test))

        fold_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_preds))
        print("#" * 25)
        print(f"### Fold {i+1} \n")
        print(f"Fold {i+1} RMSE: {fold_rmse}")
        print("#" * 25)

    pred_xgb /= FOLDS

    return model_xgb, oof_xgb, pred_xgb

# CatBoost

In [ ]:
train_cat = train.copy()
test_cat = test.copy()

In [ ]:
!pip install catboost


In [ ]:
from catboost import CatBoostRegressor

def train_CAT(train, test, model_params, features, cat_features, target):
    fix_params = {
        'task_type': "GPU",
        'iterations': 10000,
        'loss_function': 'RMSE',
        'early_stopping_rounds': 20,
    }
    model_params.update(fix_params)

    FOLDS = 10
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_cat = np.zeros(len(train))
    pred_cat = np.zeros(len(test))

    print('Training model with the following parameters')
    for k, v in model_params.items():
        print(f'{k} : {v}')

    for i, (t_idx, v_idx) in enumerate(kf.split(train)):
        X_train = train.iloc[t_idx][features].copy()
        y_train = train.iloc[t_idx][target]
        X_valid = train.iloc[v_idx][features].copy()
        y_valid = train.iloc[v_idx][target]
        X_test = test[features].copy()


        model_cat = CatBoostRegressor(**model_params)
        model_cat.fit(
            X_train, np.log1p(y_train),
            cat_features=cat_features,
            eval_set=[(X_valid, np.log1p(y_valid))],
            verbose=False
        )

        y_valid_preds = np.expm1(model_cat.predict(X_valid))
        oof_cat[v_idx] = y_valid_preds
        pred_cat += np.expm1(model_cat.predict(X_test))

        fold_rmse = mean_squared_error(y_valid, y_valid_preds, squared=False)
        print("#" * 25)
        print(f"### Fold {i + 1} \n")
        print(f"Fold {i + 1} RMSE: {fold_rmse}")
        print("#" * 25)

    pred_cat /= FOLDS

    return model_cat, oof_cat, pred_cat


random for


In [ ]:
from sklearn.ensemble import RandomForestRegressor

def train_RF(train, test, model_params, features, target):
    FOLDS = 10
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_rf = np.zeros(len(train))
    pred_rf = np.zeros(len(test))

    for i, (t_idx, v_idx) in enumerate(kf.split(train)):
        X_train = train.iloc[t_idx][features]
        y_train = train.iloc[t_idx][target]
        X_valid = train.iloc[v_idx][features]
        y_valid = train.iloc[v_idx][target]
        X_test = test[features]

        model_rf = RandomForestRegressor(**model_params)
        model_rf.fit(X_train, np.log1p(y_train))

        y_valid_preds = np.expm1(model_rf.predict(X_valid))
        oof_rf[v_idx] = y_valid_preds
        pred_rf += np.expm1(model_rf.predict(X_test))

        fold_rmse = mean_squared_error(y_valid, y_valid_preds, squared=False)
        print("#" * 25)
        print(f"### Fold {i + 1} \n")
        print(f"Fold {i + 1} RMSE: {fold_rmse}")
        print("#" * 25)

    pred_rf /= FOLDS

    return model_rf, oof_rf, pred_rf


HistGradientBoostingRegressor

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

def train_HGB(train, test, model_params, features, target):
    FOLDS = 10
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_hgb = np.zeros(len(train))
    pred_hgb = np.zeros(len(test))

    for i, (t_idx, v_idx) in enumerate(kf.split(train)):
        X_train = train.iloc[t_idx][features]
        y_train = train.iloc[t_idx][target]
        X_valid = train.iloc[v_idx][features]
        y_valid = train.iloc[v_idx][target]
        X_test = test[features]

        model_hgb = HistGradientBoostingRegressor(**model_params)
        model_hgb.fit(X_train, np.log1p(y_train))

        y_valid_preds = np.expm1(model_hgb.predict(X_valid))
        oof_hgb[v_idx] = y_valid_preds
        pred_hgb += np.expm1(model_hgb.predict(X_test))

        fold_rmse = mean_squared_error(y_valid, y_valid_preds, squared=False)
        print("#" * 25)
        print(f"### Fold {i + 1} \n")
        print(f"Fold {i + 1} RMSE: {fold_rmse}")
        print("#" * 25)

    pred_hgb /= FOLDS

    return model_hgb, oof_hgb, pred_hgb


# Ensemble

In [ ]:
# Model parameters
model_lgb_params = {
    'max_depth': 5,
    'learning_rate': 0.01,
    'colsample_bytree': 0.40481123886709114,
    'subsample': 0.7673666426617842,
    'num_leaves': 46,
    'min_child_samples': 34,
    'lambda_l1': 0.0032893870728708495,
    'lambda_l2': 1.5780171816002318e-06,
    'bagging_freq': 5,
    'cat_features': cat,
    'n_estimators': 5000,
    'objective': 'regression',
    'early_stopping_rounds': 20
}

model_xgb_params = {
    'max_depth': 4,
    'learning_rate': 0.09180807102095336,
    'colsample_bytree': 0.3809438487844099,
    'subsample': 0.844622438351228,
    'n_estimators': 419,
    'min_child_weight': 3.714743419003562,
    'reg_alpha': 5.80197653137552e-06,
    'reg_lambda': 1.2374095115455325e-08,
    'gamma': 0.0037460722016019465
}

model_xgb2_params = {
    "max_depth": 9,
    "learning_rate": 0.018203874021653552,
    "colsample_bytree": 0.41392312362600636,
    "subsample": 0.870771567534879,
    "n_estimators": 10000,
    "min_child_weight": 6.587958958652532,
    "reg_alpha": 1.675358492618636e-07,
    "reg_lambda": 0.004228750471811781,
    "gamma": 0.02009243264106564,
    "tree_method": "gpu_hist",
    "objective": "reg:squarederror",
    "early_stopping_rounds": 20,
    "eval_metric": "rmse",
    "enable_categorical": True
}

model_cat_params = {
    'depth': 5,
    'learning_rate': 0.05259516359861675,
    'bagging_temperature': 0.5,
    'l2_leaf_reg': 6.806823646372654e-06,
    'random_strength': 5.889614035287661
}

# === New models imports ===
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import numpy as np

# === Encoding for RandomForest and HistGBD ===
train_rf = train.copy()
test_rf = test.copy()

for col in features:
    if train_rf[col].dtype == "object" or str(train_rf[col].dtype).startswith('category'):
        le = LabelEncoder()
        train_rf[col] = le.fit_transform(train_rf[col].astype(str))
        test_rf[col] = le.transform(test_rf[col].astype(str))

# === Functions ===
def train_random_forest(train, test, features, target):
    FOLDS = 10
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_rf = np.zeros(len(train))
    pred_rf = np.zeros(len(test))

    for i, (t_idx, v_idx) in enumerate(kf.split(train)):
        X_train = train.iloc[t_idx][features]
        y_train = train.iloc[t_idx][target]
        X_valid = train.iloc[v_idx][features]
        y_valid = train.iloc[v_idx][target]
        X_test = test[features]

        model_rf = RandomForestRegressor(
            n_estimators=500,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
        model_rf.fit(X_train, y_train)

        y_valid_preds = model_rf.predict(X_valid)
        oof_rf[v_idx] = y_valid_preds
        pred_rf += model_rf.predict(X_test)

        fold_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_preds))
        print(f"RF Fold {i+1} RMSE: {fold_rmse}")

    pred_rf /= FOLDS
    return model_rf, oof_rf, pred_rf


def train_hist_gbdt(train, test, features, target):
    FOLDS = 10
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    oof_hist = np.zeros(len(train))
    pred_hist = np.zeros(len(test))

    for i, (t_idx, v_idx) in enumerate(kf.split(train)):
        X_train = train.iloc[t_idx][features]
        y_train = train.iloc[t_idx][target]
        X_valid = train.iloc[v_idx][features]
        y_valid = train.iloc[v_idx][target]
        X_test = test[features]

        model_hist = HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=1000,
            max_depth=6,
            random_state=42
        )
        model_hist.fit(X_train, y_train)

        y_valid_preds = model_hist.predict(X_valid)
        oof_hist[v_idx] = y_valid_preds
        pred_hist += model_hist.predict(X_test)

        fold_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_preds))
        print(f"HistGBDT Fold {i+1} RMSE: {fold_rmse}")

    pred_hist /= FOLDS
    return model_hist, oof_hist, pred_hist

# ======= Train all models ========
model_rf, oof_rf, pred_rf = train_random_forest(train_rf, test_rf, features, target)
model_hist, oof_hist, pred_hist = train_hist_gbdt(train_rf, test_rf, features, target)
model_lgb, oof_lgb, pred_lgb = train_lgbm(train_lgb, test_lgb, model_lgb_params, features, cat, target)
model_xgb, oof_xgb, pred_xgb = train_XGB(train_xgb, test_xgb, model_xgb_params, features_xgb, target)
model_xgb2, oof_xgb2, pred_xgb2 = train_XGB(train_xgb2, test_xgb2, model_xgb2_params, features, target)

# ======= Ensemble all ========
oof_ensemble = (oof_lgb + oof_xgb + oof_xgb2 + oof_rf + oof_hist) / 5
pred_ensemble = (pred_lgb + pred_xgb + pred_xgb2 + pred_rf + pred_hist) / 5

# ======= Scoring ========
y_true = train[["ID", "efs", "efs_time", "race_group"]].copy()
y_pred = train[["ID"]].copy()
y_pred["prediction"] = oof_ensemble

m = score(y_true, y_pred, "ID")
print(f"\nOverall CV for KaplanMeier = {m}")


# Submission

In [ ]:
!pip install lifelines

In [ ]:
from lifelines.utils import concordance_index

def score_cindex(y_true, y_pred, id_column='ID'):
    """
    Calculates C-Index for survival prediction.

    Parameters:
    - y_true: DataFrame with columns [ID, event, time, ...]
    - y_pred: DataFrame with columns [ID, prediction]
    - id_column: Common ID column for matching

    Returns:
    - cindex score (higher is better)
    """
    # Merge ground truth and predictions on the ID column
    merged = y_true.merge(y_pred, on=id_column, how='left')

    # Extract true event times, event indicators, and model predictions
    times = merged['efs_time']     # Time to event or censoring
    events = merged['efs']         # 1 = event occurred, 0 = censored
    preds = merged['prediction']   # Model risk scores or predicted times

    # Concordance index treats higher scores as higher risk → invert if needed
    # Here we negate preds so that higher preds imply shorter survival
    cindex = concordance_index(times, -preds, events)

    return cindex


In [ ]:
# After making oof_ensemble
y_true = train[["ID", "efs", "efs_time", "race_group"]].copy()
y_pred = train[["ID"]].copy()
y_pred["prediction"] = oof_ensemble

cindex_score = score_cindex(y_true, y_pred, "ID")
print(f"\nOverall CV (C-Index) =", cindex_score)


In [ ]:
sample_submission = pd.read_csv('/sample_submission.csv')
submission = sample_submission.copy()
submission['prediction'] = pred_ensemble*100
submission.to_csv('./submission.csv', index=False)
submission
df=pd.read_csv('./submission.csv')
df

In [ ]:
import numpy as np

# Actual values and predicted values
actual = np.array([50, 50, 50])  # Replace with your actual values
predicted = np.array([41.6, 62, 37.4])  # Replace with your predicted values

# Calculate the Mean Absolute Percentage Error (MAPE)
mape = np.mean(np.abs((actual - predicted) / actual)) * 100

# Calculate accuracy
accuracy = 100 - mape

print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
print(f"Model Accuracy: {accuracy}%")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Model performance data
data = {
    'Model': [
        'Random Forest',
        'HistGBDT',
        'LightGBM',
        'XGBoost (v1)',
        'XGBoost (v2, deeper)'
    ],
    'Mean RMSE': [
        0.1982,
        0.1946,
        0.1926,
        0.1935,
        0.1938
    ]
}

df = pd.DataFrame(data)

# Set a consistent plot style
sns.set(style="whitegrid")

# Vertical bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Mean RMSE', data=df, palette='mako')
plt.title('Model Comparison by Mean RMSE')
plt.xticks(rotation=45)
plt.ylim(0.19, 0.20)
plt.tight_layout()
plt.show()

# Horizontal bar chart (alternative view)
plt.figure(figsize=(8, 5))
sns.barplot(y='Model', x='Mean RMSE', data=df, palette='flare')
plt.title('Model Comparison by Mean RMSE')
plt.xlim(0.19, 0.20)
plt.tight_layout()
plt.show()


In [ ]:
# === CODE TO GENERATE YOUR NEW RESULTS ===
from lifelines.utils import concordance_index
from sklearn.metrics import mean_squared_error

# --- Create the ground truth data ---
y_true_df = train[["ID", "efs", "efs_time"]].copy()
y_true_times = y_true_df['efs_time']
y_true_events = y_true_df['efs']
y_actual_target = train[target] # The 'y' col (KMF probs)

# --- Create a dictionary of all your OOF predictions ---
oof_preds = {
    "LightGBM": oof_lgb,
    "XGBoost (v1)": oof_xgb,
    "XGBoost (v2)": oof_xgb2,
    "Random Forest": oof_rf,
    "HistGBDT": oof_hist,
    "Ensemble": oof_ensemble
}

# --- Calculate C-Index and RMSE for all models ---
results = []
for model_name, preds in oof_preds.items():
    # C-Index (Higher is better)
    # We negate the predictions because C-index expects a "risk score"
    # Your model predicts "survival probability" (high = good)
    # So, -preds turns it into a "risk score" (low = good)
    c_index = concordance_index(
        y_true_times,
        -preds,  # Negating the probability to create a risk score
        y_true_events
    )

    # RMSE (Lower is better)
    # Calculated against the KMF probability target 'y'
    rmse = np.sqrt(mean_squared_error(y_actual_target, preds))

    results.append({
        "Model": model_name,
        "Concordance Index (C-Index)": c_index,
        "Mean RMSE": rmse
    })

# --- Create the final results DataFrame ---
results_df = pd.DataFrame(results).sort_values(by="Concordance Index (C-Index)", ascending=False)

print(results_df.to_markdown(index=False, floatfmt=".4f"))

In [ ]:
# === CODE TO GENERATE FAIRNESS TABLE ===

# --- Add race_group to the ground truth data ---
y_true_df['race_group'] = train['race_group']
y_true_df['prediction'] = oof_ensemble # Use the best model (ensemble)

race_groups = y_true_df['race_group'].unique()
equity_results = []

for race in race_groups:
    race_df = y_true_df[y_true_df['race_group'] == race]

    race_c_index = concordance_index(
        race_df['efs_time'],
        -race_df['prediction'],
        race_df['efs']
    )

    equity_results.append({
        "Race Group": race,
        "Patient Count": len(race_df),
        "C-Index": race_c_index
    })

# --- Calculate stats ---
equity_df = pd.DataFrame(equity_results)
mean_c = equity_df['C-Index'].mean()
std_c = equity_df['C-Index'].std()
competition_score = mean_c - std_c

print(equity_df.to_markdown(index=False, floatfmt=".4f"))
print(f"\nMean C-Index: {mean_c:.4f}")
print(f"Std Dev of C-Index: {std_c:.4f}")
print(f"Competition 'Equity Score': {competition_score:.4f}") # This is your 0.685!

In [ ]:
# === CODE TO GENERATE FEATURE IMPORTANCE ===
import matplotlib.pyplot as plt
import seaborn as sns

# Get the feature importances from the last trained LGBM model
feature_importances = model_lgb.feature_importances_
feature_names = features # Use the 'features' list from your code

# Create a DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False).head(20) # Get top 20

# Plot
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Top 20 Most Important Features (LightGBM)')
plt.tight_layout()
plt.show()

print(importance_df.to_markdown(index=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = train[num].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title("Numerical Feature Correlation Heatmap")
plt.show()


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = train[num].fillna(0)
vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif.sort_values(by="VIF", ascending=False))


In [ ]:
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()
for grp in train['race_group'].unique():
    grp_df = train[train['race_group'] == grp]
    kmf.fit(grp_df['efs_time'], grp_df['efs'], label=grp)
    kmf.plot_survival_function()

plt.title("Kaplan-Meier Survival Curves by Race Group")
plt.xlabel("Time")
plt.ylabel("Survival Probability")
plt.show()


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model_lgb, train[features], np.log1p(train[target]), n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    "Feature": features,
    "Importance": perm['importances_mean']
}).sort_values(by="Importance", ascending=False).head(15)

sns.barplot(x="Importance", y="Feature", data=perm_df)
plt.title("Permutation Feature Importance (LightGBM)")
plt.show()


In [ ]:
residuals = train[target] - oof_ensemble
plt.scatter(oof_ensemble, residuals, alpha=0.4)
plt.axhline(0, color='r', linestyle='--')
plt.title("Residuals vs Predictions (Ensemble)")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.show()


In [ ]:
import shap
explainer = shap.TreeExplainer(model_lgb)
shap_values = explainer.shap_values(train[features])
shap.summary_plot(shap_values, train[features], plot_type="bar")


In [ ]:
# Make sure all features are numeric (convert categories to int codes)
train_perm = train[features].copy()

for col in train_perm.columns:
    if str(train_perm[col].dtype).startswith("category"):
        train_perm[col] = train_perm[col].cat.codes
    elif train_perm[col].dtype == "object":
        train_perm[col] = train_perm[col].astype("category").cat.codes

# Run permutation importance again
perm = permutation_importance(
    model_lgb,
    train_perm,
    np.log1p(train[target]),
    n_repeats=10,
    random_state=42
)

perm_df = pd.DataFrame({
    "Feature": features,
    "Importance": perm['importances_mean']
}).sort_values(by="Importance", ascending=False).head(15)

sns.barplot(x="Importance", y="Feature", data=perm_df, palette="viridis")
plt.title("Permutation Feature Importance (LightGBM)")
plt.tight_layout()
plt.show()


In [ ]:
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model_lgb.feature_importances_
}).sort_values(by="Importance", ascending=False).head(15)

sns.barplot(x="Importance", y="Feature", data=importance_df, palette="magma")
plt.title("Feature Importance (Built-in LightGBM)")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# --- Fix is here ---
# Use train_lgb[features] and train_lgb[target]
# This is the *same data* your model was trained on.
perm = permutation_importance(
    model_lgb,
    train_lgb[features],
    np.log1p(train_lgb[target]),
    n_repeats=10,
    random_state=42,
    n_jobs=-1  # This will make it run much faster!
)
# --- End of fix ---

perm_df = pd.DataFrame({
    "Feature": features,
    "Importance": perm['importances_mean']
}).sort_values(by="Importance", ascending=False).head(20)

# Plot
plt.figure(figsize=(10, 8))
sns.barplot(x="Importance", y="Feature", data=perm_df, palette='viridis')
plt.title("Permutation Feature Importance (LightGBM)")
plt.tight_layout()
plt.show()

print(perm_df.to_markdown(index=False))

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- 1. Prepare a purely numeric dataset ---
X_shap = train[features].copy()
for col in X_shap.columns:
    if str(X_shap[col].dtype).startswith("category"):
        X_shap[col] = X_shap[col].cat.codes
    elif X_shap[col].dtype == "object":
        X_shap[col] = X_shap[col].astype("category").cat.codes
X_shap = X_shap.fillna(0)

# --- 2. Take a manageable sample for speed ---
X_sample = X_shap.sample(300, random_state=42)

# --- 3. Define a plain prediction function (no LightGBM internals) ---
def model_predict(x):
    # Wrap predictions to handle numpy arrays directly
    df = pd.DataFrame(x, columns=X_shap.columns)
    return model_lgb.predict(df)

# --- 4. Use SHAP's model-agnostic Explainer ---
explainer = shap.Explainer(model_predict, X_sample)

# --- 5. Compute SHAP values ---
shap_values = explainer(X_sample)

# --- 6. Plot global feature importance ---
plt.title("Model-Agnostic SHAP Summary (LightGBM Ensemble)")
shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=15)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import LabelEncoder

# ---- Step 1: Select numeric and encoded categorical features ----
# Assuming `train` is your DataFrame and `features` list contains selected columns
df_vif = train[features].copy()

# Encode categorical variables numerically (important for VIF computation)
for col in df_vif.select_dtypes(include=['object', 'category']).columns:
    le = LabelEncoder()
    df_vif[col] = le.fit_transform(df_vif[col].astype(str))

# ---- Step 2: Compute VIF ----
vif_data = pd.DataFrame()
vif_data["feature"] = df_vif.columns
vif_data["VIF"] = [variance_inflation_factor(df_vif.values, i) for i in range(df_vif.shape[1])]

# ---- Step 3: Sort and filter high-VIF features ----
vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)

# ---- Step 4: Visualize ----
plt.figure(figsize=(8, 6))
sns.barplot(
    x="VIF",
    y="feature",
    data=vif_data.head(20),
    palette="viridis"
)
plt.title("Top 20 Features by Variance Inflation Factor (VIF)")
plt.xlabel("VIF Value")
plt.ylabel("Feature")
plt.tight_layout()

# ---- Step 5: Save image for LaTeX ----
plt.savefig("vif_results.png", dpi=300)
plt.show()

# ---- Step 6: Print VIF table ----
print(vif_data.head(20))


In [ ]:
!pip install numpy